In [ ]:
!pip install -q openai==0.28 pypdf faiss-cpu gradio numpy

In [ ]:
import os
import numpy as np
import pickle
import openai
import faiss
import gradio as gr
from pypdf import PdfReader

# ================== CONFIG ================== #

openai.api_key = ''  # or paste directly as string

EMBED_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
VECTOR_STORE_PATH = "vector_store.pkl"

# ================== PDF EXTRACTION (BULLETPROOF) ================== #

def extract_text_from_uploaded_pdf(file_input):
    """
    Works for:
    - string paths
    - dict objects
    - tempfile objects
    """

    if isinstance(file_input, dict):
        file_path = file_input.get("path") or file_input.get("name")
    elif isinstance(file_input, str):
        file_path = file_input
    else:
        file_path = getattr(file_input, "name", None)

    if not file_path or not os.path.exists(file_path):
        raise ValueError(f"Invalid file path received: {file_input}")

    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text, os.path.basename(file_path)

# ================== CHUNKING ================== #

def chunk_text(text, metadata, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []
    sources = []

    i = 0
    while i < len(words):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))
        sources.append(metadata)
        i += chunk_size - overlap

    return chunks, sources

# ================== EMBEDDING ================== #

def embed_texts(texts):
    response = openai.Embedding.create(
        model=EMBED_MODEL,
        input=texts
    )
    return np.array([e["embedding"] for e in response["data"]]).astype("float32")

# ================== VECTOR STORE (FAISS) ================== #

def build_faiss_index(vectors):
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(vectors)
    return index


def save_vector_store(index, chunks, sources):
    with open(VECTOR_STORE_PATH, "wb") as f:
        pickle.dump((index, chunks, sources), f)


def load_vector_store():
    with open(VECTOR_STORE_PATH, "rb") as f:
        return pickle.load(f)

# ================== RETRIEVAL ================== #

def retrieve(query, index, chunks, sources, k=4):
    query_vector = embed_texts([query])
    distances, indices = index.search(query_vector, k)

    context = []
    citations = []

    for idx in indices[0]:
        context.append(chunks[idx])
        citations.append(sources[idx])

    return "\n\n".join(context), list(set(citations))

# ================== GENERATION ================== #

def generate_answer(query, context, sources):
    response = openai.ChatCompletion.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a precise assistant. Answer strictly from the provided context."
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ],
        temperature=0.2,
        max_tokens=350
    )

    answer = response["choices"][0]["message"]["content"].strip()
    citation_text = "\n\n📚 Sources:\n" + "\n".join(set(sources))

    return answer + citation_text

# ================== INGEST ================== #

def ingest_uploaded_pdfs(uploaded_files):
    try:
        if not uploaded_files:
            return " Please upload at least one PDF."

        all_chunks = []
        all_sources = []

        for file_input in uploaded_files:
            text, filename = extract_text_from_uploaded_pdf(file_input)

            if not text.strip():
                continue

            chunked, srcs = chunk_text(text, filename)
            all_chunks.extend(chunked)
            all_sources.extend(srcs)

        if not all_chunks:
            return " No readable text found in uploaded PDFs."

        vectors = embed_texts(all_chunks)
        index = build_faiss_index(vectors)

        save_vector_store(index, all_chunks, all_sources)

        return f"Successfully ingested {len(uploaded_files)} PDFs with {len(all_chunks)} chunks."

    except Exception as e:
        return f" Ingestion Failed: {str(e)}"

# ================== QUESTION ANSWERING ================== #

def ask_question(query, top_k):
    try:
        if not os.path.exists(VECTOR_STORE_PATH):
            return "⚠️ Please ingest PDFs first."

        index, chunks, sources = load_vector_store()
        context, citations = retrieve(query, index, chunks, sources, k=top_k)
        answer = generate_answer(query, context, citations)

        return answer

    except Exception as e:
        return f" Query Failed: {str(e)}"

# ================== GRADIO UI ================== #

with gr.Blocks() as app:
    gr.Markdown("## RAG-based PDF Question Answering System")

    gr.Markdown("###  Upload PDFs")
    pdf_uploader = gr.File(
        label="Upload one or more PDFs",
        file_types=[".pdf"],
        file_count="multiple"
    )

    ingest_btn = gr.Button(" Ingest PDFs")
    ingest_status = gr.Textbox(label="Ingestion Status")

    ingest_btn.click(
        fn=ingest_uploaded_pdfs,
        inputs=[pdf_uploader],
        outputs=[ingest_status]
    )

    gr.Markdown("###  Ask a Question")

    with gr.Row():
        query_input = gr.Textbox(placeholder="Ask your question...")
        top_k_slider = gr.Slider(1, 10, value=4, label="Top-K Chunks")

    answer_output = gr.Textbox(label=" Answer", lines=12)
    ask_btn = gr.Button("Ask")

    ask_btn.click(
        fn=ask_question,
        inputs=[query_input, top_k_slider],
        outputs=[answer_output]
    )

app.launch(share=True)
